In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

In [ ]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.layers import GlobalMaxPooling2D, GlobalAveragePooling2D, Concatenate, Input
from tensorflow.keras.models import Model

# --- ✅ EuPea Attention Module ---
class EuPea(tf.keras.layers.Layer):
    def __init__(self, epsilon=1e-5):
        super(EuPea, self).__init__()
        self.epsilon = epsilon

    def call(self, x):
        x_mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        x_std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True)

        d = tf.sqrt(tf.square(x - x_mean) + self.epsilon)
        d_min = tf.reduce_min(d, axis=[1, 2], keepdims=True)
        d_max = tf.reduce_max(d, axis=[1, 2], keepdims=True)
        d_norm = (d - d_min) / (d_max - d_min + self.epsilon)

        z = tf.square((x - x_mean) / (x_std + self.epsilon))
        z_min = tf.reduce_min(z, axis=[1, 2], keepdims=True)
        z_max = tf.reduce_max(z, axis=[1, 2], keepdims=True)
        z_norm = (z - z_min) / (z_max - z_min + self.epsilon)

        attention = tf.nn.sigmoid(d_norm + z_norm)
        return x * attention

# --- ✅ Build ResNet101 + EuPea Feature Model ---
def build_eupea_resnet101_feature_model():
    base_model = ResNet101(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    input_tensor = Input(shape=(224, 224, 3))
    x = tf.keras.applications.resnet.preprocess_input(input_tensor)

    backbone = Model(
        inputs=base_model.input,
        outputs=[
            base_model.get_layer("pool1_pool").output,
            base_model.get_layer("conv2_block3_out").output,
            base_model.get_layer("conv3_block4_out").output,
            base_model.get_layer("conv4_block23_out").output,
            base_model.get_layer("conv5_block3_out").output
        ]
    )

    features = backbone(x)
    attention = EuPea()

    pooling_layers = [
        GlobalMaxPooling2D(),
        GlobalMaxPooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D()
    ]

    pooled_outputs = []
    for feat, pool in zip(features, pooling_layers):
        attn_feat = attention(feat)
        pooled_outputs.append(pool(attn_feat))

    final_feature = Concatenate()(pooled_outputs)
    return Model(inputs=input_tensor, outputs=final_feature)

# --- ✅ Instantiate Feature Extractor ---
eupea_feature_model = build_eupea_resnet101_feature_model()
print(eupea_feature_model.summary())

# --- ✅ Define directories ---
base_dir = r"/kaggle/input/datasets/saurabhsainiji/anno-casewise-train-balanced-test-untouched/Anno-casewise-spilit-train-balanced-test-untouched_/Patient-wise/Split-3"
train_dir = os.path.join(base_dir, "Training")
test_dir = os.path.join(base_dir, "Testing")
labels = ['Category-1', 'Category-2', 'Category-3']

# --- ✅ Helper function for feature extraction (with image names) ---
def extract_features_from_directory(data_dir):
    df = pd.DataFrame(columns=['features', 'labels', 'image_name'])
    for label_idx, label in enumerate(labels):
        label_dir = os.path.join(data_dir, label)
        if not os.path.exists(label_dir):
            print(f"⚠️ Warning: {label_dir} not found")
            continue

        for subdir, dirs, files in os.walk(label_dir):
            for file in files:
                if not (file.lower().endswith(".jpg") or file.lower().endswith(".png")):
                    continue

                img_path = os.path.join(subdir, file)
                img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
                x = tf.keras.preprocessing.image.img_to_array(img)
                x = tf.expand_dims(x, axis=0)

                # Extract EuPea + ResNet101 features
                features = eupea_feature_model(x)
                features_np = features.numpy().flatten()

                # Append image name, features, and label
                df.loc[len(df)] = [features_np, label_idx, file]

    return df

# --- ✅ Extract features for Train and Test sets separately ---
res_101_train_features_df = extract_features_from_directory(train_dir)
res_101_test_features_df = extract_features_from_directory(test_dir)

# --- ✅ Display summary ---
print("✅ Training features extracted:", res_101_train_features_df.shape)
print("✅ Testing features extracted:", res_101_test_features_df.shape)


In [ ]:
#print(res_50_train_features_df)

In [ ]:
# ============================================================
# ✅ 1. Extract Features & Labels from your TensorFlow DataFrame
# ============================================================

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split


class CervicalDataset(Dataset):
    def __init__(self, df):
        self.X = np.stack(df['features'].values)
        self.y = df['labels'].values.astype(int)

        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CervicalDataset(res_101_train_features_df)
test_dataset  = CervicalDataset(res_101_test_features_df)


# ============================================================
# ✅ 4. Create Data Loaders
# ============================================================

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train batches:", len(train_loader))
print("Test  batches:", len(test_loader))


In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get input dimension from training features
input_dim = len(res_101_train_features_df['features'].iloc[0])
print("Input dim:", input_dim)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FeedForwardNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 1000),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1000, 1000),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1000, num_classes)
        )

    def forward(self, x):
        return self.model(x)


model = FeedForwardNN(input_dim=input_dim, num_classes=3).to(device)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=0.0, weight=None, reduction='mean'):
        super().__init__()
        self.alpha = alpha      # balance pos/neg (scalar or per-class tensor)
        self.gamma = gamma      # focusing parameter (paper recommends 2)
        self.weight = weight    # per-class weights (like CE)
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)                          # recover prediction confidence
        focal = self.alpha * (1 - pt) ** self.gamma * ce   # apply focal modulation
        
        if self.reduction == 'mean':
            return focal.mean()
        elif self.reduction == 'sum':
            return focal.sum()
        return focal  # 'none'

criterion = FocalLoss()

In [ ]:
import torch
from torch.optim import Optimizer

In [ ]:
import torch
from torch.optim import Optimizer

class Adam(Optimizer):
    """
    Implements Adam algorithm with optional AMSGrad and Weight Decay.
    Arguments:
        params (iterable): iterable of parameters to optimize or dicts defining parameter groups
        lr (float, optional): learning rate (default: 1e-3)
        betas (Tuple[float, float], optional): coefficients used for computing
            running averages of gradient and its square (default: (0.9, 0.999))
        eps (float, optional): term added to the denominator to improve
            numerical stability (default: 1e-8)
        weight_decay (float, optional): weight decay (L2 penalty) (default: 0)
        amsgrad (bool, optional): whether to use the AMSGrad variant (default: False)
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.0, amsgrad=False):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= eps:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 0: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 1: {betas[1]}")
        if not 0.0 <= weight_decay:
            raise ValueError(f"Invalid weight_decay value: {weight_decay}")

        defaults = dict(lr=lr, betas=betas, eps=eps,
                        weight_decay=weight_decay, amsgrad=amsgrad)
        super(Adam, self).__init__(params, defaults)

    def step(self, closure=None):
        """
        Performs a single optimization step.
        Arguments:
            closure (callable, optional): A closure that reevaluates the model
                and returns the loss.
        """
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad.data
                if grad.is_sparse:
                    raise RuntimeError('Adam does not support sparse gradients')

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    # Exponential moving average of gradient values
                    state['m'] = torch.zeros_like(p.data)
                    # Exponential moving average of squared gradient values
                    state['v'] = torch.zeros_like(p.data)
                    if group['amsgrad']:
                        # Maintains the maximum of all v_t
                        state['max_v'] = torch.zeros_like(p.data)

                m, v = state['m'], state['v']
                beta1, beta2 = group['betas']
                state['step'] += 1
                t = state['step']

                # Apply weight decay (L2 regularization)
                if group['weight_decay'] != 0:
                    grad = grad.add(p.data, alpha=group['weight_decay'])

                # Update biased first moment estimate
                m.mul_(beta1).add_(grad, alpha=1 - beta1)
                # Update biased second raw moment estimate
                v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Compute bias-corrected first and second moment estimates
                m_hat = m / (1 - beta1 ** t)
                v_hat = v / (1 - beta2 ** t)

                if group['amsgrad']:
                    max_v = state['max_v']
                    torch.maximum(max_v, v_hat, out=max_v)
                    denom = max_v.sqrt().add_(group['eps'])
                else:
                    denom = v_hat.sqrt().add_(group['eps'])

                step_size = group['lr']
                p.data.addcdiv_(m_hat, denom, value=-step_size)

        return loss

In [ ]:
import copy
import pandas as pd
import numpy as np
import torch

def train_model(model, optimizer, name, num_epochs=200):
    """
    Trains a given model using the provided optimizer and tracks:
    - The model with minimum test loss (best_model_loss)
    - The model with maximum test accuracy (best_model_acc)
    """

    criterion = FocalLoss()

    train_loss_list = []
    train_acc_list = []
    test_loss_list = []
    test_acc_list = []

    # Tracking best metrics
    best_test_acc = 0.0
    best_test_loss = float('inf')

    best_model_acc = None
    best_model_loss = None

    best_acc_epoch = -1
    best_loss_epoch = -1

    for epoch in range(num_epochs):
        model.train()
        epoch_loss, correct, total = 0.0, 0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * X_batch.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

        avg_train_loss = epoch_loss / total
        train_accuracy = correct / total
        train_loss_list.append(avg_train_loss)
        train_acc_list.append(train_accuracy)

        # ---------------------------
        # Evaluation on test data
        # ---------------------------
        model.eval()
        test_loss, correct_test, total_test = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)

                test_loss += loss.item() * X_batch.size(0)
                _, preds = torch.max(outputs, 1)
                correct_test += (preds == y_batch).sum().item()
                total_test += y_batch.size(0)

        avg_test_loss = test_loss / total_test
        test_accuracy = correct_test / total_test
        test_loss_list.append(avg_test_loss)
        test_acc_list.append(test_accuracy)

        # ✅ Save model when test loss is minimum
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            best_model_loss = copy.deepcopy(model.state_dict())
            best_loss_epoch = epoch + 1  # +1 since epochs start from 0

        # ✅ Save model when test accuracy is maximum
        if test_accuracy > best_test_acc:
            best_test_acc = test_accuracy
            best_model_acc = copy.deepcopy(model.state_dict())
            best_acc_epoch = epoch + 1

        print(f"{name} | Epoch {epoch+1}/{num_epochs} - "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
              f"Test Loss: {avg_test_loss:.4f}, Test Acc: {test_accuracy:.4f}")

    # ✅ Save metrics to CSV
    metrics_df = pd.DataFrame({
        'epoch': np.arange(1, num_epochs + 1),
        'train_loss': train_loss_list,
        'train_accuracy': train_acc_list,
        'test_loss': test_loss_list,
        'test_accuracy': test_acc_list
    })
    metrics_df.to_csv(f"{name}_metrics.csv", index=False)
    print(f"[{name}] Per-epoch metrics saved to '{name}_metrics.csv'")

    # ✅ Save both best models
    torch.save(best_model_loss, f"{name}_best_model_loss.pth")
    torch.save(best_model_acc, f"{name}_best_model_acc.pth")

    print(f"\n[{name}] Summary:")
    print(f"Best model (min test loss): Epoch {best_loss_epoch} | Test Loss = {best_test_loss:.4f}")
    print(f"Best model (max test accuracy): Epoch {best_acc_epoch} | Test Accuracy = {best_test_acc:.4f}")
    print(f"Models saved as:")
    print(f"  • {name}_best_model_loss.pth")
    print(f"  • {name}_best_model_acc.pth")

    return (
        train_loss_list,
        train_acc_list,
        test_loss_list,
        test_acc_list,
        best_model_loss,
        best_model_acc,
        best_loss_epoch,
        best_acc_epoch
    )

# -------------------------------
# ✅ Initialize model and DiffGrad optimizer
# -------------------------------
model1 = FeedForwardNN(input_dim=input_dim, num_classes=3).to(device)
optimizer1 = Adam(model1.parameters(), lr=1e-4, betas=(0.9, 0.999), eps=1e-8)
# -------------------------------
# ✅ Train with DiffGrad
# -------------------------------
print("Training with Adam...")
(
    adam_train_loss,
    adam_train_acc,
    adam_test_loss,
    adam_test_acc,
    adam_best_model_loss,
    adam_best_model_acc,
    adam_best_loss_epoch,
    adam_best_acc_epoch
) = train_model(model1, optimizer1, "Adam")

# -------------------------------
# ✅ Summary of best models
# -------------------------------
print("\nTraining complete for adam")
print(f"Best model (min test loss): Epoch {adam_best_loss_epoch}")
#print(f"Best model (max test accuracy): Epoch {diffgrad_best_acc_epoch}")
print("Saved models:")
#print("  • DiffGrad_best_model_loss.pth")
#print("  • DiffGrad_best_model_acc.pth")


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
# # --------------------------------------------
# # ✅ Plotting metrics for DiffGrad only (Kaggle Safe Version)
# # --------------------------------------------
# import matplotlib.pyplot as plt
# import os

# # ✅ Define output directory for Kaggle
# output_dir = '/kaggle/working'
# os.makedirs(output_dir, exist_ok=True)

# epochs = range(1, len(adam_train_loss) + 1)

# plt.figure(figsize=(16, 10))

# # Training Loss
# plt.subplot(2, 2, 1)
# plt.plot(epochs, adam_train_loss, label='Training Loss', color='blue')
# plt.title("Training Loss")
# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.legend()

# # Training Accuracy
# plt.subplot(2, 2, 2)
# plt.plot(epochs, adam_train_acc, label='Training Accuracy', color='blue')
# plt.title("Training Accuracy")
# plt.xlabel("Epoch")
# plt.ylabel("Accuracy")
# plt.legend()

# # Testing Loss
# plt.subplot(2, 2, 3)
# plt.plot(epochs, adam_test_loss, label='Testing Loss', color='blue')
# plt.title("Testing Loss")
# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.legend()

# # Testing Accuracy
# plt.subplot(2, 2, 4)
# plt.plot(epochs, adam_test_acc, label='Testing Accuracy', color='blue')
# plt.title("Testing Accuracy")
# plt.xlabel("Epoch")
# plt.ylabel("Accuracy")
# plt.legend()

# plt.tight_layout()

# # ✅ Save figure
# save_path = os.path.join(output_dir, 'adam_metrics_combined.png')
# plt.savefig(save_path)

# plt.show()
# print(f"✅ Adam combined plots saved successfully to: {save_path}")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_best_model(best_model_state, name):
    model = FeedForwardNN(input_dim=input_dim, num_classes=3).to(device)
    model.load_state_dict(best_model_state)
    model.eval()

    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, dim=1)

            y_true_all.extend(y_batch.cpu().numpy())
            y_pred_all.extend(preds.cpu().numpy())

    # ✅ Compute all evaluation metrics
    accuracy = accuracy_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all, average='macro', zero_division=0)
    recall = recall_score(y_true_all, y_pred_all, average='macro', zero_division=0)
    f1 = f1_score(y_true_all, y_pred_all, average='macro', zero_division=0)

    # ✅ Specificity (macro average)
    cm = confusion_matrix(y_true_all, y_pred_all)
    specificity_per_class = []
    for i in range(cm.shape[0]):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        specificity = tn / (tn + fp) if (tn + fp) != 0 else 0.0
        specificity_per_class.append(specificity)
    specificity_macro = np.mean(specificity_per_class)

    # ✅ Print metrics
    print(f"\n{name} - Evaluation Metrics for Best Saved Model:")
    print(f"  Accuracy   : {accuracy * 100:.2f} %")
    print(f"  Precision  : {precision:.4f}")
    print(f"  Recall     : {recall:.4f}")
    print(f"  F1 Score   : {f1:.4f}")
    print(f"  Specificity: {specificity_macro:.4f}")

    return accuracy * 100  # returning % accuracy like before

# --------------------------------------------
# ✅ Evaluate both saved best DiffGrad models
# --------------------------------------------

# Evaluate model saved based on minimum test loss
adam_final_acc_loss = evaluate_best_model(
    adam_best_model_loss, "Adam_best_loss"
)


# --------------------------------------------
# ✅ Print results
# --------------------------------------------
print("\n=== Final Test Accuracies of Best Saved Models ===")
print(f"DiffGrad Accuracy:     {adam_final_acc_loss:.4f} %")
#print(f"DiffGrad (Best Test Accuracy Model): {diffgrad_final_acc_acc:.4f} %")
